# Rheology of Newtonian & Non-Newtonian Materials

Apply the `RheologyPipeline` (from `rheology_pipeline_core.py`) to the six datasets in this folder and visualise the rheology of each sample **step-by-step**:

| # | Dataset       | Material family             | Expected behaviour                  |
|---|---------------|-----------------------------|-------------------------------------|
| 1 | `silicone.csv`| Silicone oils (calibration) | Newtonian (single-RPM per cell)     |
| 2 | `PEG.csv`     | Poly-ethylene glycol        | Newtonian → mildly shear-thinning   |
| 3 | `sepineo.csv` | Sepineo (acrylate gel)      | Shear-thinning                      |
| 4 | `solagum.csv` | Solagum (polysaccharide)    | Strongly shear-thinning             |
| 5 | `carbopol.csv`| Carbopol 980 (yield-stress) | Shear-thinning (yield-pseudoplastic)|
| 6 | `TIMS.csv`    | TIMS (new sample)           | To be inferred from flow curve      |

The pipeline:
1. Fits each drag profile $D(h)=A/(h+h_c)+B$ to extract the amplitude $A$.
2. Converts $A \to \mu_\text{app}$ via the silicone calibration $A=k\,\mu^p$.
3. If multiple RPMs are present per cell, fits a power-law $A(\dot\gamma)=A_0\,\dot\gamma^{\,n-1}$ to classify the flow regime.

## Step 1 — Imports, paths & plotting style

In [ ]:
import sys, re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# -- Make the repo importable -------------------------------------------------
REPO = Path.cwd().resolve().parent if Path.cwd().name == "analysis" else Path.cwd().resolve()  # repository root
sys.path.insert(0, str(REPO / "analysis"))
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from rheology_pipeline_core import RheologyPipeline, fit_drag_profile

# -- Dataset paths ------------------------------------------------------------
DATA_DIR  = REPO / "data" / "06_materials"
CALIB_CSV = REPO / "data" / "03_silicone_calibration" / "height_normalized.csv"

SAMPLES = {
    "silicone": DATA_DIR / "silicone.csv",
    "PEG"     : DATA_DIR / "peg.csv",
    "sepineo" : DATA_DIR / "sepineo.csv",
    "solagum" : DATA_DIR / "solagum.csv",
    "carbopol": DATA_DIR / "carbopol.csv",
    "tims"    : DATA_DIR / "tims.csv",
}

# -- Google plotting style ----------------------------------------------------
mpl.rcParams["font.family"] = "Arial"
GOOGLE_COLORS = ["#4285F4", "#EA4335", "#FBBC05", "#34A853", "#FF6D00", "#46BDC6"]
GOOGLE_CMAP   = LinearSegmentedColormap.from_list(
    "google", [GOOGLE_COLORS[0], GOOGLE_COLORS[2], GOOGLE_COLORS[1]]
)
BORDER, FS_LABEL, FS_TICK, FS_LEGEND = 1.5, 20, 14, 12

# Central map for stable figure numbering. Edit these IDs any time before export.
FIGURE_IDS = {
    "sample_silicone_analysis": 101,
    "silicone_parity": 102,
    "silicone_stress": 103,
    "sample_peg_analysis": 104,
    "peg_stress": 105,
    "sample_sepineo_analysis": 106,
    "sepineo_stress": 107,
    "sample_solagum_analysis": 108,
    "solagum_stress": 109,
    "sample_carbopol_analysis": 110,
    "carbopol_stress": 111,
    "sample_tims_analysis": 112,
    "syy_stress": 113,
    "hy500_stress": 114,
    "sekisui_b41_stress": 115,
    "master_flow_curve": 116,
    "drag_silicone": 117,
    "drag_peg": 118,
    "drag_sepineo": 119,
    "drag_solagum": 120,
    "drag_carbopol": 121,
    "drag_syy": 122,
    "drag_sekisui_b41": 123,
    "drag_hy500": 124,
}

_FIGURE_COUNTER = max(FIGURE_IDS.values()) + 1 if FIGURE_IDS else 1

# Figure IDs requested for SVG export
FIG_IDS_TO_SAVE = [116, 117, 118, 119, 120, 121, 122, 123, 124, 103, 105, 107, 109, 111, 113, 114, 115]
id_to_key = {v: k for k, v in FIGURE_IDS.items()}
FIG_KEYS_TO_SAVE = [id_to_key.get(v, f"figure_{v}") for v in FIG_IDS_TO_SAVE]
FIG_ID_SET_TO_SAVE = set(FIG_IDS_TO_SAVE)
EXPORT_DIR = REPO / "analysis" / "outputs"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
SELECTED_FIGSIZE_IN = (6, 5)

def _save_selected_figures_to_svg():
    saved = []
    original_bbox = mpl.rcParams.get("savefig.bbox", None)
    mpl.rcParams["savefig.bbox"] = None
    for fid in FIG_IDS_TO_SAVE:
        if fid not in plt.get_fignums():
            continue
        fig = plt.figure(fid)
        key = id_to_key.get(fid, f"figure_{fid}")
        out = EXPORT_DIR / f"{fid:03d}_{key}.svg"
        fig.savefig(out, format="svg", bbox_inches=None, pad_inches=0)
        saved.append(out.name)
    mpl.rcParams["savefig.bbox"] = original_bbox
    if saved:
        print(f"Saved {len(saved)} SVG figure(s) -> {EXPORT_DIR}")

if hasattr(plt, "_svg_save_original_show"):
    _orig_show = plt._svg_save_original_show
else:
    _orig_show = plt.show
    plt._svg_save_original_show = _orig_show

def _show_with_svg_export(*args, **kwargs):
    _save_selected_figures_to_svg()
    return _orig_show(*args, **kwargs)

plt.show = _show_with_svg_export

def get_figure_id(figure_key: str | None = None, figure_id: int | None = None) -> int:
    """Resolve figure ID from explicit ID or key; auto-allocates if key is new."""
    global _FIGURE_COUNTER

    if figure_id is not None:
        return int(figure_id)
    if figure_key is not None:
        if figure_key not in FIGURE_IDS:
            FIGURE_IDS[figure_key] = _FIGURE_COUNTER
            _FIGURE_COUNTER += 1
        return int(FIGURE_IDS[figure_key])

    fid = _FIGURE_COUNTER
    _FIGURE_COUNTER += 1
    return int(fid)

def new_subplots(*args, figure_key: str | None = None, figure_id: int | None = None, **kwargs):
    """Create subplots with a deterministic matplotlib figure number."""
    fid = get_figure_id(figure_key=figure_key, figure_id=figure_id)
    if fid in FIG_ID_SET_TO_SAVE and "figsize" not in kwargs:
        kwargs["figsize"] = SELECTED_FIGSIZE_IN
    fig, ax = plt.subplots(*args, num=fid, **kwargs)
    fig.set_label(f"Figure {fid}")
    return fig, ax, fid

def _gs(ax):
    ax.grid(False)
    ax.tick_params(labelsize=FS_TICK)
    for sp in ax.spines.values():
        sp.set_linewidth(BORDER)

def _cb(mappable, ax, label):
    cb = plt.colorbar(mappable, ax=ax)
    cb.set_label(label, fontsize=FS_LEGEND)
    cb.ax.tick_params(labelsize=FS_TICK - 2)
    cb.outline.set_linewidth(BORDER)
    return cb

print("Repo            :", REPO)
print("Calibration CSV :", CALIB_CSV.relative_to(REPO), "(exists:", CALIB_CSV.exists(), ")")
print("Sample CSVs     :")
for k, p in SAMPLES.items():
    print(f"  {k:<9s} -> {p.relative_to(REPO)}  (exists: {p.exists()})")
print(f"Figure ID presets loaded: {len(FIGURE_IDS)}")
print(f"SVG export targets (IDs): {FIG_IDS_TO_SAVE}")
print(f"SVG export directory      : {EXPORT_DIR}")

## Step 2 — Load the silicone calibration

Load the universal cone-plate offset $h_c$ and the silicone calibration $A = k\,\mu^p$ from `results/Auto-runs/height_normalized.csv`. All six samples are analysed with the **same** calibration so their viscosities are directly comparable.

In [ ]:
pipeline = RheologyPipeline()
cal = pipeline.load_silicone_calibration(CALIB_CSV)

print("=== Universal calibration ===")
for k, v in cal.items():
    print(f"  {k:25s}: {v:.6g}")

print()
print(f"h_c (universal) = {pipeline.H_C_UNIVERSAL:.4f} mm")
print(f"k               = {pipeline.SILICONE_K:.4e}")
print(f"p               = {pipeline.SILICONE_P:.4f}")
print(f"Calibrated?     = {pipeline.is_calibrated}")

## Step 3 — Generic helper: load, clean and analyse a sample CSV

Every dataset follows the same row schema (`Cell_Label, Z_Height_mm, RPM, Torque_%`). The helpers below:

1. **`load_sample(path)`** — read CSV, drop repeated headers / NaNs / `Torque_%==0` rows, coerce numeric columns and **re-zero the height per cell** ($h = h_\text{raw} - h_\text{raw,min}$, as our repo convention dictates).
2. **`parse_true_mu(label)`** — extract a reference viscosity from the cell label whenever it is encoded (e.g. `l60000_b_a69080` → 69 080 cP, `980carb0.3%_49810` → 49 810 cP).
3. **`analyse_cell(df_cell, pipeline)`** — auto-detect single vs multi-RPM, call `pipeline.predict_rheology(...)`, and additionally extract per-RPM amplitudes / apparent viscosities for plotting the flow curve.

In [ ]:
# -- Generic loader -----------------------------------------------------------
NUM_COLS = ["Z_Height_mm", "RPM", "Torque_%", "Rotational_Drag"]

def load_sample(path: Path) -> pd.DataFrame:
    """Read a sample CSV; clean it; re-zero height; collapse duplicate samples."""
    sample_key = path.stem.lower()
    use_blank_sep = sample_key in {"peg", "carbopol", "sepineo", "solagum", "tims"}
    df = pd.read_csv(
        path,
        encoding="latin-1",
        low_memory=False,
        skip_blank_lines=not use_blank_sep,
    )

    if use_blank_sep:
        # In PEG/carbopol/sepineo/solagum/TIMS files, each empty row separates experiments.
        blank_sep = df[["Cell_Label", "Z_Height_mm", "RPM", "Torque_%"]].isna().all(axis=1)
        df["Experiment_ID"] = blank_sep.cumsum().astype(int)
    else:
        df["Experiment_ID"] = 0

    # Some CSVs contain accidentally-concatenated header rows.
    df = df[df["RPM"].astype(str).str.strip() != "RPM"].copy()

    for c in NUM_COLS:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df = df.dropna(subset=NUM_COLS)
    df = df[df["Torque_%"] > 0]            # ignore "insufficient_data" rows
    df = df[df["RPM"] > 0]

    if use_blank_sep:
        # Keep experiments separated even when the base label is repeated.
        df["Cell_Label"] = (
            df["Cell_Label"].astype(str).str.strip()
            + " | exp"
            + (df["Experiment_ID"] + 1).astype(str)
        )

    # Re-zero height *per cell* - convention from the repo notes.
    df["h_mm"] = df.groupby("Cell_Label")["Z_Height_mm"].transform(
        lambda s: s - s.min()
    )

    # Round RPM to a stable key (avoids float-precision multi-grouping)
    df["RPM_key"] = df["RPM"].round(4)

    # Collapse duplicate samples at the same (cell, RPM, h_mm) by mean Torque.
    df = (df.groupby(["Cell_Label", "RPM_key", "h_mm"], as_index=False)
            .agg(Z_Height_mm=("Z_Height_mm", "mean"),
                 RPM        =("RPM",         "mean"),
                 **{"Torque_%": ("Torque_%", "mean")},
                 Rotational_Drag=("Rotational_Drag", "mean"),
                 Experiment_ID=("Experiment_ID", "first")))
    return df.reset_index(drop=True)


# -- Label -> labelled viscosity (cP) ----------------------------------------
_RE_AFTER_A = re.compile(r"_a([\d.]+)")     # silicone:  ..._a69080  or _a69.08
_RE_TRAIL   = re.compile(r"_(\d+(?:\.\d+)?)$")  # PEG/sepineo/solagum/carbopol: trailing _<num>

def _strip_experiment_suffix(label: str) -> str:
    return str(label).split(" | exp", 1)[0].strip()

def parse_true_mu(label: str) -> float:
    """Extract labelled viscosity (cP) from a Cell_Label, or NaN if not present."""
    if not isinstance(label, str) or label in ("", "Cell_Label", "nan"):
        return float("nan")
    label = _strip_experiment_suffix(label)

    m = _RE_AFTER_A.search(label)
    if m and m.group(1).strip("."):
        val = float(m.group(1))
        # Silicone files mix units: integers are cP, decimals are Pa*s (x 1000 -> cP)
        return val * 1000.0 if "." in m.group(1) else val

    m = _RE_TRAIL.search(label)
    if m:
        return float(m.group(1))

    return float("nan")


# -- Per-cell analysis --------------------------------------------------------
def analyse_cell(df_cell: pd.DataFrame, pipeline) -> dict:
    """
    Run the rheology pipeline on a single cell (one Cell_Label).
    Returns dict: {result, per_rpm, groups, label}.
    """
    label = df_cell["Cell_Label"].iloc[0]
    groups = []  # list of (rpm, h_arr, T_arr)
    for rpm_key, sub in df_cell.groupby("RPM_key", sort=True):
        sub = sub.sort_values("h_mm")
        if len(sub) >= 4:
            groups.append((float(sub["RPM"].mean()),
                           sub["h_mm"].to_numpy(),
                           sub["Torque_%"].to_numpy()))

    if not groups:
        return dict(label=label, result={"regime": "no-data", "n": np.nan,
                                         "mu_app_cP": np.nan, "fit_quality": np.nan,
                                         "n_rpms": 0},
                    per_rpm=pd.DataFrame(), groups=[])

    rpms = [g[0] for g in groups]
    hs   = [g[1] for g in groups]
    Ts   = [g[2] for g in groups]

    if len(groups) == 1:
        result = pipeline.predict_rheology(hs[0], Ts[0], rpms[0])
    else:
        result = pipeline.predict_rheology(hs, Ts, rpms)

    # Also compute per-RPM amplitude / mu_app for plotting the flow curve.
    per_rpm_rows = []
    for rpm, h, T in groups:
        D = T / float(rpm)
        fit = fit_drag_profile(h, D, h_c=pipeline.H_C_UNIVERSAL)
        A = fit["A"]
        mu = float(pipeline.amplitude_to_viscosity(np.array([A]))[0]) \
             if np.isfinite(A) and A > 0 else np.nan
        per_rpm_rows.append(dict(
            RPM=rpm,
            gamma_dot_1_s=float(pipeline.geo.shear_rate(rpm)),
            A=A, B=fit["B"], h_c=fit["h_c"], R2=fit["R2"],
            mu_app_cP=mu,
        ))
    per_rpm = pd.DataFrame(per_rpm_rows).sort_values("RPM").reset_index(drop=True)

    return dict(label=label, result=result, per_rpm=per_rpm, groups=groups)


# -- Per-sample analysis + plots ----------------------------------------------
def _drag_model(h, A, B, h_c):
    return A / (h + h_c) + B


def analyse_sample(name: str,
                   path: Path,
                   pipeline,
                   max_cells_to_plot: int | None = None,
                   figure_key: str | None = None,
                   figure_id: int | None = None) -> pd.DataFrame:
    """
    Analyse every cell in a sample CSV, plot drag profiles + flow curves,
    and return a tidy per-cell summary DataFrame.
    """
    df = load_sample(path)
    cells = list(df["Cell_Label"].dropna().unique())
    print(f"=== {name}  -  {len(cells)} cell(s)  ·  {len(df)} rows ===")

    cell_analyses = {}
    summary_rows  = []
    for cl in cells:
        an = analyse_cell(df[df["Cell_Label"] == cl], pipeline)
        cell_analyses[cl] = an
        r = an["result"]
        r2_drag = an["per_rpm"]["R2"].mean() if not an["per_rpm"].empty else np.nan
        summary_rows.append(dict(
            sample      = name,
            label       = cl,
            true_mu_cP  = parse_true_mu(cl),
            n_rpms      = int(len(an["groups"])),
            regime      = r.get("regime", ""),
            n_idx       = r.get("n", np.nan),
            mu_app_cP   = r.get("mu_app_cP", r.get("mu_app_cP_at_min_gamma_dot", np.nan)),
            R2_drag_mean= r2_drag,
            R2_powerlaw = r.get("R2_powerlaw", np.nan),
            Cell_Label  = cl,
        ))
    summary = pd.DataFrame(summary_rows)

    # -- Choose which cells to plot --------------------------------------------
    plot_cells = cells
    if max_cells_to_plot is not None and len(plot_cells) > max_cells_to_plot:
        # Spread across the labelled-viscosity range for a representative subset.
        s = summary.dropna(subset=["true_mu_cP"]).sort_values("true_mu_cP")
        if len(s) >= max_cells_to_plot:
            idx = np.linspace(0, len(s) - 1, max_cells_to_plot).round().astype(int)
            plot_cells = s.iloc[idx]["Cell_Label"].tolist()
        else:
            plot_cells = cells[:max_cells_to_plot]

    multi_rpm_sample = any(len(cell_analyses[c]["groups"]) > 1 for c in plot_cells)
    ncols = 2 if multi_rpm_sample else 1
    nrows = len(plot_cells)
    fk = figure_key if figure_key is not None else f"sample_{str(name).lower()}_analysis"
    fig, axes, fid = new_subplots(
        nrows, ncols,
        figsize=(6.0 * ncols, 3.6 * nrows),
        squeeze=False,
        figure_key=fk,
        figure_id=figure_id,
    )

    for i, cl in enumerate(plot_cells):
        an = cell_analyses[cl]
        groups, per_rpm, res = an["groups"], an["per_rpm"], an["result"]

        # -- Drag-profile panel ------------------------------------------------
        axD = axes[i, 0]
        rpms_arr = np.array([g[0] for g in groups])
        norm = mpl.colors.Normalize(vmin=rpms_arr.min(), vmax=rpms_arr.max()) \
               if len(rpms_arr) > 1 else None
        for j, (rpm, h, T) in enumerate(groups):
            D = T / rpm
            color = GOOGLE_CMAP(norm(rpm)) if norm is not None else GOOGLE_COLORS[0]
            axD.scatter(h, D, s=40, color=color, edgecolors="k", linewidths=0.4,
                        alpha=0.9, zorder=3, label=f"{rpm:g} RPM")
            row = per_rpm.iloc[j]
            if np.isfinite(row["A"]):
                hh = np.linspace(h.min(), h.max(), 200)
                axD.plot(hh, _drag_model(hh, row["A"], row["B"], row["h_c"]),
                         "-", color=color, lw=1.6, alpha=0.85)
        true_mu = parse_true_mu(cl)
        mu_pred = res.get("mu_app_cP", res.get("mu_app_cP_at_min_gamma_dot", np.nan))
        title = f"{cl}"
        if np.isfinite(true_mu):
            title += f"   |   labelled {true_mu:,.0f} cP"
        if np.isfinite(mu_pred):
            title += f"   ·   pred {mu_pred:,.0f} cP"
        axD.set_title(title, fontsize=FS_LABEL - 2)
        axD.set_xlabel("Re-zeroed height  h (mm)", fontsize=FS_LABEL - 2)
        axD.set_ylabel("Drag  D = T(%) / RPM", fontsize=FS_LABEL - 2)
        if len(rpms_arr) <= 6:
            axD.legend(fontsize=FS_LEGEND - 2, frameon=False, loc="best")
        _gs(axD)

        # -- Flow-curve panel (multi-RPM only) ---------------------------------
        if multi_rpm_sample:
            axF = axes[i, 1]
            pr = per_rpm.dropna(subset=["mu_app_cP"])
            pr = pr[pr["mu_app_cP"] > 0]
            if len(pr) >= 2:
                axF.plot(pr["gamma_dot_1_s"], pr["mu_app_cP"], "-o",
                         color=GOOGLE_COLORS[1], lw=2.0, ms=7,
                         markeredgecolor="k", markeredgewidth=0.4, label="data")
                n_idx = res.get("n", np.nan)
                if np.isfinite(n_idx) and "K_Pas_n" in res \
                   and np.isfinite(res["K_Pas_n"]):
                    g = np.geomspace(pr["gamma_dot_1_s"].min(),
                                     pr["gamma_dot_1_s"].max(), 100)
                    K_cP = res["K_Pas_n"] / pipeline.geo.CP_TO_PAS
                    mu_pl = K_cP * g ** (n_idx - 1.0)
                    axF.plot(g, mu_pl, "--", color="k", lw=1.4,
                             label=f"power-law fit  n={n_idx:.2f}")
                axF.set_xscale("log"); axF.set_yscale("log")
            elif len(pr) == 1:
                axF.scatter(pr["gamma_dot_1_s"], pr["mu_app_cP"],
                            s=70, color=GOOGLE_COLORS[1], edgecolors="k")
                axF.set_xscale("log"); axF.set_yscale("log")
            axF.set_xlabel(r"Shear rate  $\dot\gamma$ (1/s)", fontsize=FS_LABEL - 2)
            axF.set_ylabel("mu_app (cP)", fontsize=FS_LABEL - 2)
            axF.set_title(f"Flow curve · regime: {res.get('regime', '?')}",
                          fontsize=FS_LABEL - 2)
            axF.legend(fontsize=FS_LEGEND - 2, frameon=False, loc="best")
            _gs(axF)

    plt.tight_layout(); plt.show()
    print(f"Figure ID [{fk}] = {fid}")

    with pd.option_context("display.float_format", "{:,.3g}".format,
                           "display.max_rows", 60,
                           "display.width", 140):
        cols = ["label", "true_mu_cP", "n_rpms", "regime",
                "n_idx", "mu_app_cP", "R2_drag_mean", "R2_powerlaw"]
        print(summary[cols].to_string(index=False))
    return summary

## Step 4 — Sample 1 · Silicone oils (Newtonian reference)

Each silicone cell was measured at a **single RPM**, so the pipeline returns a Newtonian viscosity per cell. The label encodes the rheometer reference (e.g. `l60000_b_a69080` → 69 080 cP). We expect the predicted $\mu_\text{app}$ to track the label very closely — this is essentially the calibration self-check. We plot a handful of representative cells across the full viscosity range.

In [ ]:
# silicone has 37 cells -> plot only a handful of representative ones,
# but still tabulate the full set.
summary_silicone = analyse_sample(
    "silicone",
    SAMPLES["silicone"],
    pipeline,
    max_cells_to_plot=6,
    figure_key="sample_silicone_analysis",
)

# -- Parity plot: predicted vs labelled viscosity -----------------------------
sil = summary_silicone.dropna(subset=["true_mu_cP", "mu_app_cP"]).copy()
sil = sil[(sil["true_mu_cP"] > 0) & (sil["mu_app_cP"] > 0)]

if not sil.empty:
    fig, ax, fig_id = new_subplots(figsize=(6, 6), figure_key="silicone_parity")
    lim = [sil["true_mu_cP"].min() * 0.6, sil["true_mu_cP"].max() * 1.6]
    ax.plot(lim, lim, "k--", lw=2, label="ideal 1:1")
    ax.fill_between(lim, [v*0.8 for v in lim], [v*1.2 for v in lim],
                    alpha=0.10, color=GOOGLE_COLORS[0], label="+-20 %")
    ax.scatter(sil["true_mu_cP"], sil["mu_app_cP"],
               s=80, color=GOOGLE_COLORS[1], edgecolors="k",
               linewidths=0.5, zorder=3)
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlim(lim);     ax.set_ylim(lim)
    ax.set_xlabel("Labelled  mu  (cP)",  fontsize=FS_LABEL)
    ax.set_ylabel("Predicted mu_app (cP)", fontsize=FS_LABEL)
    ax.set_title("Silicone parity check", fontsize=FS_LABEL)
    ax.legend(fontsize=FS_LEGEND, frameon=False)
    _gs(ax)
    plt.tight_layout(); plt.show()
    print(f"Figure ID [silicone_parity] = {fig_id}")

    err = (sil["mu_app_cP"] - sil["true_mu_cP"]) / sil["true_mu_cP"] * 100
    print(f"  median |error| = {err.abs().median():.1f} %"
          f"   within +-20% : {(err.abs() <= 20).mean()*100:.0f} %"
          f"   N = {len(sil)}")

In [ ]:
# Build a shared stress table used by all sample-specific stress plots.
stress_rows = []
for sample_name, sample_path in SAMPLES.items():
    df_sample = load_sample(sample_path)
    if df_sample.empty:
        continue

    for cl in df_sample["Cell_Label"].dropna().unique():
        an = analyse_cell(df_sample[df_sample["Cell_Label"] == cl], pipeline)
        per = an["per_rpm"].copy()
        if per.empty:
            continue

        per = per[np.isfinite(per["gamma_dot_1_s"]) & np.isfinite(per["mu_app_cP"])]
        per = per[(per["gamma_dot_1_s"] > 0) & (per["mu_app_cP"] > 0)]
        if per.empty:
            continue

        per["tau_Pa"] = (per["mu_app_cP"] * pipeline.geo.CP_TO_PAS) * per["gamma_dot_1_s"]
        per["sample"] = sample_name
        per["label"] = cl
        per = per.rename(columns={"R2": "R2_drag"})
        stress_rows.append(
            per[["sample", "label", "RPM", "gamma_dot_1_s", "mu_app_cP", "tau_Pa", "R2_drag"]]
        )

stress_df = pd.concat(stress_rows, ignore_index=True) if stress_rows else pd.DataFrame(
    columns=["sample", "label", "RPM", "gamma_dot_1_s", "mu_app_cP", "tau_Pa", "R2_drag"]
)
print(f"Built shear-stress table: {len(stress_df)} (cell, RPM) points across {stress_df['sample'].nunique()} samples")


def fit_power_law(gamma_dot: np.ndarray, tau: np.ndarray):
    """Fit tau = K * gamma_dot^n in log space; return (K, n, R2)."""
    x = np.asarray(gamma_dot, dtype=float)
    y = np.asarray(tau, dtype=float)
    keep = np.isfinite(x) & np.isfinite(y) & (x > 0) & (y > 0)
    x, y = x[keep], y[keep]
    if len(x) < 2:
        return np.nan, np.nan, np.nan

    lx = np.log(x)
    ly = np.log(y)
    n, logK = np.polyfit(lx, ly, 1)
    yhat = n * lx + logK
    ss_res = float(np.sum((ly - yhat) ** 2))
    ss_tot = float(np.sum((ly - np.mean(ly)) ** 2))
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else 1.0
    return float(np.exp(logK)), float(n), float(r2)


def clean_for_powerlaw(d: pd.DataFrame,
                       r2_drag_min: float = 0.2,
                       mad_k: float = 4.5,
                       min_keep: int = 3,
                       max_iter: int = 4):
    """Return a robust mask for points suitable for power-law fitting."""
    keep = (
        np.isfinite(d["gamma_dot_1_s"])
        & np.isfinite(d["tau_Pa"])
        & (d["gamma_dot_1_s"] > 0)
        & (d["tau_Pa"] > 0)
    )

    if "R2_drag" in d.columns:
        keep = keep & (d["R2_drag"].fillna(1.0) >= r2_drag_min)

    keep = keep.astype(bool).to_numpy()

    for _ in range(max_iter):
        if keep.sum() <= min_keep:
            break

        x = d.loc[keep, "gamma_dot_1_s"].to_numpy(dtype=float)
        y = d.loc[keep, "tau_Pa"].to_numpy(dtype=float)
        K, n, _ = fit_power_law(x, y)
        if not (np.isfinite(K) and np.isfinite(n)):
            break

        log_res = np.log(y) - (np.log(K) + n * np.log(x))
        med = np.median(log_res)
        mad = np.median(np.abs(log_res - med))
        if not np.isfinite(mad) or mad <= 1e-12:
            break

        good_local = np.abs(log_res - med) <= (mad_k * mad)
        if good_local.all():
            break

        keep_idx = np.flatnonzero(keep)
        keep[keep_idx[~good_local]] = False

    return keep


PALETTE = GOOGLE_COLORS.copy()
PEG_COLORS = {
    "5%300K_577": GOOGLE_COLORS[0],
    "10%300K_7158": GOOGLE_COLORS[1],
    "5%600K_3254": GOOGLE_COLORS[2],
    "6.5%600K_4109": GOOGLE_COLORS[3],
    "10%600K_64250": GOOGLE_COLORS[4],
}

In [ ]:
# Silicone-only shear stress vs shear rate (single panel, Google style)
sample = "silicone"
sub = stress_df[stress_df["sample"] == sample].copy()

if sub.empty:
    print("No silicone data found in stress_df.")
else:
    # Pick representative silicone cells across the labelled-viscosity range
    sil_rank = summary_silicone.dropna(subset=["true_mu_cP"]).sort_values("true_mu_cP")
    n_show = min(6, max(1, len(sil_rank)))
    if not sil_rank.empty:
        pick_idx = np.linspace(0, len(sil_rank) - 1, n_show).round().astype(int)
        picked_labels = sil_rank.iloc[pick_idx]["label"].tolist()
    else:
        picked_labels = list(sub["label"].dropna().unique())[:n_show]

    # Color-map by labelled viscosity for a smooth Google-style gradient
    mu_vals = np.array([parse_true_mu(lb) for lb in picked_labels], dtype=float)
    finite_mu = np.isfinite(mu_vals) & (mu_vals > 0)
    if finite_mu.sum() >= 2:
        norm_mu = mpl.colors.LogNorm(vmin=float(np.nanmin(mu_vals[finite_mu])),
                                     vmax=float(np.nanmax(mu_vals[finite_mu])))
    else:
        norm_mu = None

    fig, ax, fig_id = new_subplots(figsize=(6, 5), figure_key="silicone_stress")
    fit_gmax = float(sub["gamma_dot_1_s"].max())

    for i, lb in enumerate(picked_labels):
        d = (sub[sub["label"] == lb]
             .sort_values("gamma_dot_1_s")
             .reset_index(drop=True))

        d = d[np.isfinite(d["gamma_dot_1_s"]) & np.isfinite(d["tau_Pa"])
              & (d["gamma_dot_1_s"] > 0) & (d["tau_Pa"] > 0)]
        if d.empty:
            continue

        mu_label = parse_true_mu(lb)
        if norm_mu is not None and np.isfinite(mu_label) and mu_label > 0:
            color = GOOGLE_CMAP(norm_mu(mu_label))
        else:
            color = GOOGLE_COLORS[i % len(GOOGLE_COLORS)]

        # Silicone cells are often single-point in stress_df; use Newtonian slope through origin.
        if len(d) >= 2:
            K_fit, n_fit, R2_fit = fit_power_law(
                d["gamma_dot_1_s"].to_numpy(),
                d["tau_Pa"].to_numpy(),
            )
        else:
            g1 = float(d["gamma_dot_1_s"].iloc[0])
            t1 = float(d["tau_Pa"].iloc[0])
            K_fit = t1 / g1
            n_fit = 1.0
            R2_fit = 1.0

        ax.scatter(
            d["gamma_dot_1_s"],
            d["tau_Pa"],
            s=100,
            color=color,
            edgecolors="k",
            linewidths=0.4,
            alpha=0.95,
            zorder=4,
        )

        gg = np.geomspace(0.1, fit_gmax, 200)
        ax.plot(
            gg,
            K_fit * gg ** n_fit,
            "--",
            color=color,
            lw=2.0,
            alpha=0.95,
            label=rf"$\tau={K_fit:.3g}\,\dot\gamma^{{{n_fit:.1f}}}$"
                  if np.isfinite(mu_label) else
                  rf"{lb}: $\tau={K_fit:.3g}\,\dot\gamma^{{{n_fit:.1f}}}$",
        )

    ax.set_xlim(0, 50)
    ax.set_ylim(0, 500)
    ax.set_xlabel(r"Shear rate  $\dot\gamma$  (1/s)", fontsize=FS_LABEL)
    ax.set_ylabel(r"Shear stress  $\tau$  (Pa)", fontsize=FS_LABEL)
    ax.legend(fontsize=FS_LEGEND, frameon=False, loc="best")
    _gs(ax)
    plt.tight_layout()
    plt.show()
    print(f"Figure ID [silicone_stress] = {fig_id}")

## Step 5 — Sample 2 · PEG (poly-ethylene glycol)

PEG solutions are essentially Newtonian at low concentrations and develop mild shear-thinning at higher MW/concentration. Every PEG cell was probed at **multiple RPMs**, so the pipeline now fits a power-law $A(\dot\gamma)=A_0\,\dot\gamma^{n-1}$ and classifies each formulation.

In [ ]:
summary_PEG = analyse_sample(
    "PEG",
    SAMPLES["PEG"],
    pipeline,
    figure_key="sample_peg_analysis",
)

In [ ]:
# PEG-only shear stress vs shear rate (single panel, Google style)
sample = "PEG"
sub = stress_df[stress_df["sample"] == sample].copy()

if sub.empty:
    print("No PEG data found in stress_df.")
else:
    peg_targets = [
        ("PEG-300K 5%",   "5%300K_577"),
        ("PEG-300K 10%",  "10%300K_7158"),
        ("PEG-600K 5%",   "5%600K_3254"),
        ("PEG-600K 6.5%", "6.5%600K_4109"),
        ("PEG-600K 10%",  "10%600K_64250"),
    ]

    fig, ax, fig_id = new_subplots(figsize=(6, 5), figure_key="peg_stress")
    fit_gmax = float(sub["gamma_dot_1_s"].max())

    labels_all = sorted(sub["label"].astype(str).unique())
    for name, base_label in peg_targets:
        color = PEG_COLORS.get(base_label, GOOGLE_COLORS[0])

        # Merge all experiment variants for the same PEG formulation
        candidates = [
            lb for lb in labels_all
            if lb == base_label or lb.startswith(f"{base_label} | exp")
        ]

        if not candidates:
            print(f"[warn] No matching label found for {base_label}")
            continue

        d = (sub[sub["label"].isin(candidates)]
             .sort_values("gamma_dot_1_s")
             .reset_index(drop=True))

        # Stage 1: strict cleaning
        keep = clean_for_powerlaw(d)
        d_in = d[keep]

        # Stage 2: relaxed cleaning if strict dropped too much
        if len(d_in) < 2:
            keep_relaxed = clean_for_powerlaw(
                d, r2_drag_min=0.0, mad_k=8.0, min_keep=2, max_iter=2
            )
            d_in = d[keep_relaxed]

        # Stage 3: raw finite positive points as last resort
        if len(d_in) < 2:
            raw_mask = (
                np.isfinite(d["gamma_dot_1_s"])
                & np.isfinite(d["tau_Pa"])
                & (d["gamma_dot_1_s"] > 0)
                & (d["tau_Pa"] > 0)
            )
            d_in = d[raw_mask]

        if len(d_in) < 2:
            print(f"[warn] No usable points found for {base_label}")
            continue

        K_fit, n_fit, R2_fit = fit_power_law(
            d_in["gamma_dot_1_s"].to_numpy(),
            d_in["tau_Pa"].to_numpy(),
        )
        if not (np.isfinite(K_fit) and np.isfinite(n_fit)):
            print(f"[warn] Fit failed for {base_label}")
            continue

        ax.scatter(
            d_in["gamma_dot_1_s"],
            d_in["tau_Pa"],
            s=100,
            color=color,
            edgecolors="k",
            linewidths=0.4,
            alpha=0.95,
            zorder=4,
        )

        gg = np.geomspace(0.001, fit_gmax, 200)
        ax.plot(
            gg,
            K_fit * gg ** n_fit,
            "--",
            color=color,
            lw=2.0,
            alpha=0.95,
            label=rf"{name}: $\tau={K_fit:.3g}\,\dot\gamma^{{{n_fit:.2f}}}$  ($R^2$={R2_fit:.2f})",
        )

    ax.set_xlim(0, 200)
    ax.set_ylim(0, 1000)
    ax.set_xlabel(r"Shear rate  $\dot\gamma$  (1/s)", fontsize=FS_LABEL)
    ax.set_ylabel(r"Shear stress  $\tau$  (Pa)", fontsize=FS_LABEL)
    ax.legend(fontsize=FS_LEGEND-0.75, frameon=False, loc="best")
    _gs(ax)
    plt.tight_layout()
    plt.show()
    print(f"Figure ID [peg_stress] = {fig_id}")

## Step 6 — Sample 3 · Sepineo (cross-linked acrylate gel)

Sepineo is a classic shear-thinning thickener used in skincare. Expect $n < 1$ and $\mu_\text{app}$ that decreases with shear rate.

In [ ]:
summary_sepineo = analyse_sample(
    "sepineo",
    SAMPLES["sepineo"],
    pipeline,
    figure_key="sample_sepineo_analysis",
)

In [ ]:
# Sepineo-only shear stress vs shear rate (single panel, Google style)
sample = "sepineo"
sub = stress_df[stress_df["sample"] == sample].copy()

if sub.empty:
    print("No sepineo data found in stress_df.")
else:
    # Match by concentration prefix so labels still resolve if trailing IDs change
    sepineo_targets = [
        ("Sepineo 1%",   "1%sep",   GOOGLE_COLORS[0]),
        ("Sepineo 1.5%", "1.5%sep", GOOGLE_COLORS[1]),
        ("Sepineo 2%",   "2%sep",   GOOGLE_COLORS[3]),
    ]

    fig, ax, fig_id = new_subplots(figsize=(6, 5), figure_key="sepineo_stress")
    fit_gmax = float(sub["gamma_dot_1_s"].max())

    labels_all = sorted(sub["label"].astype(str).unique())
    for name, label_prefix, color in sepineo_targets:
        candidates = [
            lb for lb in labels_all
            if lb == label_prefix
            or lb.startswith(f"{label_prefix}_")
            or lb.startswith(f"{label_prefix} | exp")
            or lb.startswith(f"{label_prefix}_") and " | exp" in lb
        ]

        if not candidates:
            print(f"[warn] No matching label found for prefix {label_prefix}")
            continue

        best = None
        for lb in candidates:
            d = (sub[sub["label"] == lb]
                 .sort_values("gamma_dot_1_s")
                 .reset_index(drop=True))

            # Stage 1: strict cleaning
            keep = clean_for_powerlaw(d)
            d_in = d[keep]

            # Stage 2: relaxed cleaning if strict dropped too much
            if len(d_in) < 2:
                keep_relaxed = clean_for_powerlaw(
                    d, r2_drag_min=0.0, mad_k=8.0, min_keep=2, max_iter=2
                )
                d_in = d[keep_relaxed]

            # Stage 3: raw finite positive points as last resort
            if len(d_in) < 2:
                raw_mask = (
                    np.isfinite(d["gamma_dot_1_s"]).astype(bool)
                    & np.isfinite(d["tau_Pa"]).astype(bool)
                    & (d["gamma_dot_1_s"] > 0)
                    & (d["tau_Pa"] > 0)
                )
                d_in = d[raw_mask]

            if len(d_in) < 2:
                continue

            K_fit, n_fit, R2_fit = fit_power_law(
                d_in["gamma_dot_1_s"].to_numpy(),
                d_in["tau_Pa"].to_numpy(),
            )
            if not (np.isfinite(K_fit) and np.isfinite(n_fit)):
                continue

            score = R2_fit if np.isfinite(R2_fit) else -np.inf
            if best is None or score > best["score"]:
                best = {
                    "label": lb,
                    "d_in": d_in,
                    "K": K_fit,
                    "n": n_fit,
                    "R2": R2_fit,
                    "score": score,
                }

        if best is None:
            print(f"[warn] No usable points found for prefix {label_prefix}")
            continue

        d_in = best["d_in"]
        K_fit, n_fit, R2_fit = best["K"], best["n"], best["R2"]

        ax.scatter(
            d_in["gamma_dot_1_s"],
            d_in["tau_Pa"],
            s=100,
            color=color,
            edgecolors="k",
            linewidths=0.4,
            alpha=0.95,
            zorder=4,
        )

        gg = np.geomspace(0.001, fit_gmax, 200)
        ax.plot(
            gg,
            K_fit * gg ** n_fit,
            "--",
            color=color,
            lw=2.0,
            alpha=0.95,
            label=rf"{name}: $\tau={K_fit:.3g}\,\dot\gamma^{{{n_fit:.2f}}}$  ($R^2$={R2_fit:.2f})",
        )

    ax.set_xlim(0, 50)
    ax.set_ylim(0, 300)
    ax.set_xlabel(r"Shear rate  $\dot\gamma$  (1/s)", fontsize=FS_LABEL)
    ax.set_ylabel(r"Shear stress  $\tau$  (Pa)", fontsize=FS_LABEL)
    ax.legend(fontsize=FS_LEGEND, frameon=False, loc="best")
    _gs(ax)
    plt.tight_layout()
    plt.show()
    print(f"Figure ID [sepineo_stress] = {fig_id}")

## Step 7 — Sample 4 · Solagum (polysaccharide gum)

Solagum gels strongly with concentration; expect a steep drop of $\mu_\text{app}$ versus $\dot\gamma$ ($n \ll 1$).

In [ ]:
summary_solagum = analyse_sample(
    "solagum",
    SAMPLES["solagum"],
    pipeline,
    figure_key="sample_solagum_analysis",
)

In [ ]:
# Solagum-only shear stress vs shear rate (single panel, Google style)
sample = "solagum"
sub = stress_df[stress_df["sample"] == sample].copy()

if sub.empty:
    print("No solagum data found in stress_df.")
else:
    # Match by concentration prefix so labels still resolve if trailing IDs change
    solagum_targets = [
        ("Solagum 1%",   "1%sola",   GOOGLE_COLORS[0]),
        ("Solagum 2%",   "2%sola",   GOOGLE_COLORS[1]),
        ("Solagum 2.5%", "2.5%sola", GOOGLE_COLORS[3]),
        ("Solagum 3%",   "3%sola",   GOOGLE_COLORS[2]),
    ]

    fig, ax, fig_id = new_subplots(figsize=(6, 5), figure_key="solagum_stress")
    fit_gmax = float(sub["gamma_dot_1_s"].max())

    labels_all = sorted(sub["label"].astype(str).unique())
    for name, label_prefix, color in solagum_targets:
        # Accept exact prefix label and any experiment suffix variant
        candidates = [
            lb for lb in labels_all
            if lb == label_prefix
            or lb.startswith(f"{label_prefix}_")
            or lb.startswith(f"{label_prefix} | exp")
            or lb.startswith(f"{label_prefix}_") and " | exp" in lb
        ]

        if not candidates:
            print(f"[warn] No matching label found for prefix {label_prefix}")
            continue

        best = None
        for lb in candidates:
            d = (sub[sub["label"] == lb]
                 .sort_values("gamma_dot_1_s")
                 .reset_index(drop=True))

            # Stage 1: strict cleaning
            keep = clean_for_powerlaw(d)
            d_in = d[keep]

            # Stage 2: relaxed cleaning if strict dropped too much
            if len(d_in) < 2:
                keep_relaxed = clean_for_powerlaw(
                    d, r2_drag_min=0.0, mad_k=8.0, min_keep=2, max_iter=2
                )
                d_in = d[keep_relaxed]

            # Stage 3: raw finite positive points as last resort
            if len(d_in) < 2:
                raw_mask = (
                    np.isfinite(d["gamma_dot_1_s"])
                    & np.isfinite(d["tau_Pa"])
                    & (d["gamma_dot_1_s"] > 0)
                    & (d["tau_Pa"] > 0)
                )
                d_in = d[raw_mask]

            if len(d_in) < 2:
                continue

            K_fit, n_fit, R2_fit = fit_power_law(
                d_in["gamma_dot_1_s"].to_numpy(),
                d_in["tau_Pa"].to_numpy(),
            )
            if not (np.isfinite(K_fit) and np.isfinite(n_fit)):
                continue

            score = R2_fit if np.isfinite(R2_fit) else -np.inf
            if best is None or score > best["score"]:
                best = {
                    "label": lb,
                    "d_in": d_in,
                    "K": K_fit,
                    "n": n_fit,
                    "R2": R2_fit,
                    "score": score,
                }

        if best is None:
            print(f"[warn] No usable points found for prefix {label_prefix}")
            continue

        d_in = best["d_in"]
        K_fit, n_fit, R2_fit = best["K"], best["n"], best["R2"]

        # Scatter data points used in fit
        ax.scatter(
            d_in["gamma_dot_1_s"],
            d_in["tau_Pa"],
            s=100,
            color=color,
            edgecolors="k",
            linewidths=0.4,
            alpha=0.95,
            zorder=4,
        )

        # Fitted power-law line: tau = K * gamma_dot^n
        gg = np.geomspace(0.001, fit_gmax, 200)
        ax.plot(
            gg,
            K_fit * gg ** n_fit,
            "--",
            color=color,
            lw=2.0,
            alpha=0.95,
            label=rf"{name}: $\tau={K_fit:.3g}\,\dot\gamma^{{{n_fit:.2f}}}$  ($R^2$={R2_fit:.2f})",
        )

    ax.set_xlim(0, 100)
    ax.set_ylim(0, 500)
    ax.set_xlabel(r"Shear rate  $\dot\gamma$  (1/s)", fontsize=FS_LABEL)
    ax.set_ylabel(r"Shear stress  $\tau$  (Pa)", fontsize=FS_LABEL)
    ax.legend(fontsize=FS_LEGEND, frameon=False, loc="best")
    _gs(ax)
    plt.tight_layout()
    plt.show()
    print(f"Figure ID [solagum_stress] = {fig_id}")

## Step 8 — Sample 5 · Carbopol 980 (yield-pseudoplastic)

Carbopol exhibits a yield stress plus strong shear-thinning. The power-law fit captures the high-shear branch ($n \ll 1$); the yield contribution shows up as the up-turn of $\mu_\text{app}$ at the lowest shear rates.

In [ ]:
summary_carbopol = analyse_sample(
    "carbopol",
    SAMPLES["carbopol"],
    pipeline,
    figure_key="sample_carbopol_analysis",
)

In [ ]:
# Carbopol-only shear stress vs shear rate (single panel, Google style)
sample = "carbopol"
sub = stress_df[stress_df["sample"] == sample].copy()

if sub.empty:
    print("No carbopol data found in stress_df.")
else:
    carbopol_targets = [
        ("Carbopol 0.2%", "980carb0.2%", GOOGLE_COLORS[0]),
        ("Carbopol 0.3%", "980carb0.3%", GOOGLE_COLORS[1]),
        ("Carbopol 0.35%", "980carb0.35%", GOOGLE_COLORS[2]),
        ("Carbopol 0.4%", "980carb0.4%", GOOGLE_COLORS[3]),
    ]

    fig, ax, fig_id = new_subplots(figsize=(6, 5), figure_key="carbopol_stress")
    fit_gmax = float(sub["gamma_dot_1_s"].max())

    labels_all = sorted(sub["label"].astype(str).unique())
    for name, label_prefix, color in carbopol_targets:
        candidates = [
            lb for lb in labels_all
            if lb == label_prefix
            or lb.startswith(f"{label_prefix}_")
            or lb.startswith(f"{label_prefix} | exp")
            or lb.startswith(f"{label_prefix}_") and " | exp" in lb
        ]

        if not candidates:
            print(f"[warn] No matching label found for prefix {label_prefix}")
            continue

        best = None
        for lb in candidates:
            d = (sub[sub["label"] == lb]
                 .sort_values("gamma_dot_1_s")
                 .reset_index(drop=True))

            # Stage 1: strict cleaning
            keep = clean_for_powerlaw(d)
            d_in = d[keep]

            # Stage 2: relaxed cleaning if strict dropped too much
            if len(d_in) < 2:
                keep_relaxed = clean_for_powerlaw(
                    d, r2_drag_min=0.0, mad_k=8.0, min_keep=2, max_iter=2
                )
                d_in = d[keep_relaxed]

            # Stage 3: raw finite positive points as last resort
            if len(d_in) < 2:
                raw_mask = (
                    np.isfinite(d["gamma_dot_1_s"]).astype(bool)
                    & np.isfinite(d["tau_Pa"]).astype(bool)
                    & (d["gamma_dot_1_s"] > 0)
                    & (d["tau_Pa"] > 0)
                )
                d_in = d[raw_mask]

            if len(d_in) < 2:
                continue

            K_fit, n_fit, R2_fit = fit_power_law(
                d_in["gamma_dot_1_s"].to_numpy(),
                d_in["tau_Pa"].to_numpy(),
            )
            if not (np.isfinite(K_fit) and np.isfinite(n_fit)):
                continue

            score = R2_fit if np.isfinite(R2_fit) else -np.inf
            if best is None or score > best["score"]:
                best = {
                    "label": lb,
                    "d_in": d_in,
                    "K": K_fit,
                    "n": n_fit,
                    "R2": R2_fit,
                    "score": score,
                }

        if best is None:
            print(f"[warn] No usable points found for prefix {label_prefix}")
            continue

        d_in = best["d_in"]
        K_fit, n_fit, R2_fit = best["K"], best["n"], best["R2"]

        ax.scatter(
            d_in["gamma_dot_1_s"],
            d_in["tau_Pa"],
            s=100,
            color=color,
            edgecolors="k",
            linewidths=0.4,
            alpha=0.95,
            zorder=4,
        )

        gg = np.geomspace(0.001, fit_gmax, 200)
        ax.plot(
            gg,
            K_fit * gg ** n_fit,
            "--",
            color=color,
            lw=2.0,
            alpha=0.95,
            label=rf"{name}: $\tau={K_fit:.3g}\,\dot\gamma^{{{n_fit:.2f}}}$  ($R^2$={R2_fit:.2f})",
        )

    ax.set_xlim(0, 3)
    ax.set_ylim(0, 50)
    ax.set_xlabel(r"Shear rate  $\dot\gamma$  (1/s)", fontsize=FS_LABEL)
    ax.set_ylabel(r"Shear stress  $\tau$  (Pa)", fontsize=FS_LABEL)
    ax.legend(fontsize=FS_LEGEND, frameon=False, loc="best")
    _gs(ax)
    plt.tight_layout()
    plt.show()
    print(f"Figure ID [carbopol_stress] = {fig_id}")

## Step 8.5 - Sample 6 · SYY (raw CSV)

Read the raw `syy.csv` dataset directly and generate a sample-only shear-stress panel with power-law overlays for representative SYY formulation(s).

In [ ]:
summary_tims = analyse_sample(
    "tims",
    SAMPLES["tims"],
    pipeline,
    figure_key="sample_tims_analysis",
)

def build_stress_points_from_raw(csv_path, sample_name):
    raw = pd.read_csv(csv_path, encoding="latin-1", low_memory=False, skip_blank_lines=False)
    needed = ["Cell_Label", "Z_Height_mm", "RPM", "Torque_%"]
    missing = [c for c in needed if c not in raw.columns]
    if missing:
        raise ValueError(f"{sample_name}: missing required columns {missing}")

    blank_sep = raw[needed].isna().all(axis=1)
    raw["Experiment_ID"] = blank_sep.cumsum().astype(int)

    raw = raw[raw["RPM"].astype(str).str.strip() != "RPM"].copy()
    raw["Z_Height_mm"] = pd.to_numeric(raw["Z_Height_mm"], errors="coerce")
    raw["RPM"] = pd.to_numeric(raw["RPM"], errors="coerce")
    raw["Torque_%"] = pd.to_numeric(raw["Torque_%"], errors="coerce")

    raw = raw.dropna(subset=needed)
    raw = raw[(raw["RPM"] > 0) & (raw["Torque_%"] > 0)].copy()

    raw["Cell_Label"] = (
        raw["Cell_Label"].astype(str).str.strip()
        + " | exp"
        + (raw["Experiment_ID"] + 1).astype(str)
)
    raw["h_mm"] = raw.groupby("Cell_Label")["Z_Height_mm"].transform(lambda s: s - s.min())
    raw["RPM_key"] = raw["RPM"].round(4)

    grp = (
        raw.groupby(["Cell_Label", "RPM_key", "h_mm"], as_index=False)
        .agg(RPM=("RPM", "mean"), Torque_pct=("Torque_%", "mean"))
    )

    rows = []
    for (lb, rpm_key), d in grp.groupby(["Cell_Label", "RPM_key"], sort=True):
        d = d.sort_values("h_mm")
        if len(d) < 4:
            continue

        rpm = float(d["RPM"].mean())
        h = d["h_mm"].to_numpy(dtype=float)
        drag = d["Torque_pct"].to_numpy(dtype=float) / rpm

        fit = fit_drag_profile(h, drag, h_c=pipeline.H_C_UNIVERSAL)
        A_raw = fit.get("A", np.nan)

        if np.isfinite(A_raw):
            A_eff = abs(float(A_raw))
        else:
            B0 = float(np.nanmin(drag))
            A_eff = float(np.nanmedian((drag - B0) * (h + pipeline.H_C_UNIVERSAL)))
            A_eff = abs(A_eff)

        if not np.isfinite(A_eff):
            continue

        A_eff = max(A_eff, 1e-9)
        mu_app_cP = float(pipeline.amplitude_to_viscosity(np.array([A_eff]))[0])
        gamma = float(pipeline.geo.shear_rate(rpm))
        tau = (mu_app_cP * pipeline.geo.CP_TO_PAS) * gamma

        rows.append({
            "label": lb,
            "RPM": rpm,
            "gamma_dot_1_s": gamma,
            "tau_Pa": tau,
            "R2_drag": float(fit.get("R2", np.nan)),
        })

    return pd.DataFrame(rows)

def plot_sample_stress_from_raw(
    csv_path,
    sample_name,
    label_hint=None,
    figure_key: str | None = None,
    figure_id: int | None = None,
):
    sub = build_stress_points_from_raw(csv_path, sample_name)
    if sub.empty:
        print(f"No {sample_name} data found for stress plot.")
        return

    labels = sorted(sub["label"].unique())
    if label_hint:
        picked_labels = [lb for lb in labels if label_hint.lower() in lb.lower()]
    else:
        picked_labels = []

    if not picked_labels:
        label_rank = sub.groupby("label")["RPM"].nunique().sort_values(ascending=False)
        picked_labels = label_rank.head(1).index.tolist()

    fk = figure_key if figure_key is not None else f"{str(sample_name).lower()}_stress_raw"
    fig, ax, fid = new_subplots(figsize=(6, 5), figure_key=fk, figure_id=figure_id)
    fit_gmax = float(sub["gamma_dot_1_s"].max())

    for i, lb in enumerate(picked_labels):
        d = (
            sub[sub["label"] == lb]
            .sort_values("gamma_dot_1_s")
            .drop_duplicates(subset=["RPM"])
            .reset_index(drop=True)
        )

        if len(d) < 2:
            print(f"[warn] Not enough points for {sample_name} / {lb}")
            continue

        K_fit, n_fit, R2_fit = fit_power_law(
            d["gamma_dot_1_s"].to_numpy(),
            d["tau_Pa"].to_numpy(),
        )

        color = GOOGLE_COLORS[i % len(GOOGLE_COLORS)]
        ax.scatter(
            d["gamma_dot_1_s"],
            d["tau_Pa"],
            s=100,
            color=GOOGLE_COLORS[2],
            edgecolors="k",
            linewidths=0.4,
            alpha=0.95,
            zorder=4,
        )

        gg = np.geomspace(0.001, fit_gmax, 200)
        ax.plot(
            gg,
            K_fit * gg ** n_fit,
            "--",
            color=color,
            lw=2.0,
            alpha=0.95,
            label=rf"{sample_name}: $\tau={K_fit:.3g}\,\dot\gamma^{{{n_fit:.2f}}}$  ($R^2$={R2_fit:.2f})",
        )

    ax.set_xlabel(r"Shear rate  $\dot\gamma$  (1/s)", fontsize=FS_LABEL)
    ax.set_ylabel(r"Shear stress  $\tau$  (Pa)", fontsize=FS_LABEL)
    ax.legend(fontsize=FS_LEGEND, frameon=False, loc="best")
    _gs(ax)
    plt.tight_layout()
    plt.show()
    print(f"Figure ID [{fk}] = {fid}")

SYY_CSV = str(DATA_DIR / "syy.csv")
plot_sample_stress_from_raw(
    SYY_CSV,
    sample_name="SYY",
    label_hint="syy",
    figure_key="syy_stress",
)

In [ ]:
HY500_CSV = str(DATA_DIR / "hy500.csv")
SEKISUI_B41_CSV = str(DATA_DIR / "sekisui_b41.csv")

plot_sample_stress_from_raw(
    HY500_CSV,
    sample_name="HY-500",
    label_hint="hy",
    figure_key="hy500_stress",
)
plot_sample_stress_from_raw(
    SEKISUI_B41_CSV,
    sample_name="Sekisui-B41",
    label_hint="sekisui",
    figure_key="sekisui_b41_stress",
)

## Step 9 — Cross-sample summary

A single tidy table of every cell across all six materials, followed by a master flow-curve plot that overlays one representative cell per non-Newtonian sample so the rheology fingerprints can be compared at a glance.

In [ ]:
summary_all = pd.concat(
    [summary_silicone, summary_PEG, summary_sepineo, summary_solagum, summary_carbopol, summary_tims],
    ignore_index=True,
)

print(f"Total cells analysed: {len(summary_all)}")
print()

with pd.option_context("display.float_format", "{:,.3g}".format,
                       "display.max_rows", 200,
                       "display.width", 140):
    cols = ["sample", "label", "true_mu_cP", "n_rpms",
            "regime", "n_idx", "mu_app_cP", "R2_drag_mean", "R2_powerlaw"]
    print(summary_all[cols].to_string(index=False))

# -- Master flow-curve comparison: one representative per requested sample
def _candidate_cells(df: pd.DataFrame):
    return df["Cell_Label"].dropna().astype(str).unique().tolist()

def _candidate_raw_labels(df_raw: pd.DataFrame):
    return df_raw["label"].dropna().astype(str).unique().tolist()

fig, ax, fig_id = new_subplots(figsize=(6, 5), figure_key="master_flow_curve")
rng = np.random.default_rng(42)  # fixed seed -> stable "random" representatives
x_line = np.geomspace(0.01, 1000.0, 300)
CP_TO_KCP = 1e-3
colors = {
    "silicone": GOOGLE_COLORS[0],
    "PEG": GOOGLE_COLORS[1],
    "sepineo": GOOGLE_COLORS[2],
    "solagum": GOOGLE_COLORS[3],
    "carbopol": GOOGLE_COLORS[4],
    "SYY": GOOGLE_COLORS[5],
    "HY-500": "#8E24AA",
    "Sekisui-B41": "#00ACC1",
}
DISPLAY_NAMES = {"sepineo": "Sepineo", "solagum": "Solagum", "carbopol": "Carbopol"}

plotted_samples = []

# 1-5) Pipeline-backed samples
pipeline_samples = ["silicone", "PEG", "sepineo", "solagum", "carbopol"]
for sample in pipeline_samples:
    df = load_sample(SAMPLES[sample])
    labels = _candidate_cells(df)
    if not labels:
        print(f"[warn] No cells found for {sample}")
        continue

    if sample == "silicone":
        # User-requested silicone representative: target viscosity 20.6 Pa.s = 20,600 cP.
        target_mu_cp = 20600.0
        scored = []
        for lb in labels:
            mu_lb = parse_true_mu(lb)
            if np.isfinite(mu_lb) and mu_lb > 0:
                scored.append((abs(mu_lb - target_mu_cp), lb, mu_lb))

        if scored:
            scored.sort(key=lambda t: t[0])
            order_labels = [t[1] for t in scored]
            print(f"Silicone target 20.6 Pa.s -> selected {order_labels[0]} (label mu={scored[0][2]:.0f} cP)")
        else:
            order_labels = labels
            print("[warn] Could not parse silicone label viscosities; falling back to random silicone selection")
    else:
        order = rng.permutation(len(labels))
        order_labels = [labels[int(i)] for i in order]

    plotted = False

    for cl in order_labels:
        an = analyse_cell(df[df["Cell_Label"] == cl], pipeline)
        pr = an["per_rpm"].copy()
        pr = pr[np.isfinite(pr["gamma_dot_1_s"]) & np.isfinite(pr["mu_app_cP"])]
        pr = pr[(pr["gamma_dot_1_s"] > 0) & (pr["mu_app_cP"] > 0)]
        if pr.empty:
            continue

        pr = pr.sort_values("gamma_dot_1_s").drop_duplicates(subset=["RPM"])
        color = colors[sample]

        # Plot representative points.
        ax.scatter(
            pr["gamma_dot_1_s"],
            pr["mu_app_cP"] * CP_TO_KCP,
            s=46,
            color=color,
            edgecolors="k",
            linewidths=0.35,
            alpha=0.95,
            zorder=4,
        )

        if sample == "silicone":
            # Requested behavior: horizontal line through the representative silicone point(s).
            mu_ref = float(np.nanmedian(pr["mu_app_cP"].to_numpy(dtype=float))) * CP_TO_KCP
            if not np.isfinite(mu_ref) or mu_ref <= 0:
                continue

            # Add four extra silicone scatter points at requested shear rates with matching viscosity.
            extra_gamma = np.array([1.0, 10.0, 20.0, 25.0, 30.0], dtype=float)
            ax.scatter(
                extra_gamma,
                np.full_like(extra_gamma, mu_ref),
                s=46,
                color=color,
                edgecolors="k",
                linewidths=0.35,
                alpha=0.95,
                zorder=4,
            )

            y_line = np.full_like(x_line, mu_ref, dtype=float)
            ax.plot(
                x_line,
                y_line,
                "-",
                color=color,
                lw=2.2,
                alpha=0.95,
                label=f"Silicone",
            )
        else:
            tau = (pr["mu_app_cP"] * pipeline.geo.CP_TO_PAS) * pr["gamma_dot_1_s"]
            K_fit, n_fit, R2_fit = fit_power_law(
                pr["gamma_dot_1_s"].to_numpy(dtype=float),
                tau.to_numpy(dtype=float),
            )
            if not (np.isfinite(K_fit) and np.isfinite(n_fit)):
                continue
            mu_line = (K_fit * (x_line ** (n_fit - 1.0))) / pipeline.geo.CP_TO_PAS
            mu_line = np.where(np.isfinite(mu_line) & (mu_line > 0), mu_line, np.nan) * CP_TO_KCP
            ax.plot(
                x_line,
                mu_line,
                "-",
                color=color,
                lw=2.0,
                alpha=0.95,
                label=f"{DISPLAY_NAMES.get(sample, sample)} (n={n_fit:.2f})",
            )

        plotted_samples.append(sample)
        plotted = True
        break

    if not plotted:
        print(f"[warn] Could not find a plottable representative for {sample}")

# 6-8) Raw-only samples
raw_samples = [
    #("SYY", SYY_CSV),
    #("HY-500", HY500_CSV),
    #("Sekisui-B41", SEKISUI_B41_CSV),
]

for sample_name, csv_path in raw_samples:
    sub = build_stress_points_from_raw(csv_path, sample_name)
    if sub.empty:
        print(f"[warn] No raw stress points found for {sample_name}")
        continue

    labels = _candidate_raw_labels(sub)
    if not labels:
        print(f"[warn] No labels found for {sample_name}")
        continue

    order = rng.permutation(len(labels))
    plotted = False

    for idx in order:
        lb = labels[int(idx)]
        d = (
            sub[sub["label"] == lb]
            .sort_values("gamma_dot_1_s")
            .drop_duplicates(subset=["RPM"])
            .reset_index(drop=True)
        )
        d = d[(d["gamma_dot_1_s"] > 0) & (d["tau_Pa"] > 0)]
        if d.empty:
            continue

        mu_app = d["tau_Pa"] / d["gamma_dot_1_s"] / pipeline.geo.CP_TO_PAS
        d = d.assign(mu_app_cP=mu_app)
        d = d[np.isfinite(d["mu_app_cP"]) & (d["mu_app_cP"] > 0)]
        if d.empty:
            continue

        K_fit, n_fit, R2_fit = fit_power_law(
            d["gamma_dot_1_s"].to_numpy(dtype=float),
            d["tau_Pa"].to_numpy(dtype=float),
        )
        if not (np.isfinite(K_fit) and np.isfinite(n_fit)):
            continue

        color = colors[sample_name]
        ax.scatter(
            d["gamma_dot_1_s"],
            d["mu_app_cP"] * CP_TO_KCP,
            s=46,
            color=color,
            edgecolors="k",
            linewidths=0.35,
            alpha=0.95,
            zorder=4,
        )

        mu_line = (K_fit * (x_line ** (n_fit - 1.0))) / pipeline.geo.CP_TO_PAS
        mu_line = np.where(np.isfinite(mu_line) & (mu_line > 0), mu_line, np.nan) * CP_TO_KCP
        ax.plot(
            x_line,
            mu_line,
            "-",
            color=color,
            lw=2.0,
            alpha=0.95,
            label=f"{sample_name} (n={n_fit:.2f})",
        )

        plotted_samples.append(sample_name)
        plotted = True
        break

    if not plotted:
        print(f"[warn] Could not find a plottable random representative for {sample_name}")

ax.set_xscale("log")
ax.set_xlim(0.1, 1000.0)
ax.set_ylim(0, 50)
ax.set_xlabel(r"Shear rate  $\dot\gamma$  (1/s)", fontsize=FS_LABEL)
ax.set_ylabel(r"Apparent viscosity  $\mu_\mathrm{app}$  (KcP)", fontsize=FS_LABEL)
ax.legend(fontsize=FS_LEGEND, frameon=False, loc="best")
_gs(ax)
plt.tight_layout(); plt.show()
print(f"Figure ID [master_flow_curve] = {fig_id}")
print(f"Samples plotted in Figure 116: {len(plotted_samples)} -> {sorted(plotted_samples)}")

# -- Save the unified summary CSV next to the data ----------------------------
out_csv = EXPORT_DIR / "rheology_pipeline_summary.csv"
summary_all.to_csv(out_csv, index=False)
print(f"\nSummary written -> {out_csv.relative_to(REPO)}")

## Step 9.5 — Drag vs height (single plot per sample)

For each sample, including TIMS, plot drag $D = \mathrm{Torque}(\%)/\mathrm{RPM}$ versus re-zeroed height $h$ on one panel using the same Google-style visual language.

In [ ]:
# Helper to draw one drag-vs-height panel for any sample or direct CSV path
def plot_drag_vs_height_sample(
    sample: str | None = None,
    max_series: int = 12,
    csv_path: str | Path | None = None,
    display_name: str | None = None,
    figure_key: str | None = None,
    figure_id: int | None = None,
):
    if csv_path is None:
        if sample is None:
            raise ValueError("Provide either sample or csv_path")
        df = load_sample(SAMPLES[sample]).copy()
        plot_name = display_name or sample
    else:
        path = Path(csv_path)
        plot_name = display_name or path.stem
        raw = pd.read_csv(path, encoding="latin-1", low_memory=False, skip_blank_lines=False)
        needed = ["Cell_Label", "Z_Height_mm", "RPM", "Torque_%"]
        missing = [c for c in needed if c not in raw.columns]
        if missing:
            raise ValueError(f"{plot_name}: missing required columns {missing}")

        blank_sep = raw[needed].isna().all(axis=1)
        raw["Experiment_ID"] = blank_sep.cumsum().astype(int)

        raw = raw[raw["RPM"].astype(str).str.strip() != "RPM"].copy()
        raw["Z_Height_mm"] = pd.to_numeric(raw["Z_Height_mm"], errors="coerce")
        raw["RPM"] = pd.to_numeric(raw["RPM"], errors="coerce")
        raw["Torque_%"] = pd.to_numeric(raw["Torque_%"], errors="coerce")

        raw = raw.dropna(subset=needed)
        raw = raw[(raw["RPM"] > 0) & (raw["Torque_%"] > 0)].copy()

        raw["Cell_Label"] = (
            raw["Cell_Label"].astype(str).str.strip()
            + " | exp"
            + (raw["Experiment_ID"] + 1).astype(str)
)
        raw["h_mm"] = raw.groupby("Cell_Label")["Z_Height_mm"].transform(lambda s: s - s.min())
        raw["RPM_key"] = raw["RPM"].round(4)
        df = raw.reset_index(drop=True)

    if df.empty:
        print(f"No rows for sample: {plot_name}")
        return

    # Drag definition used throughout the notebook
    df["drag"] = df["Torque_%"] / df["RPM"]
    grp = (df.groupby(["Cell_Label", "RPM_key", "h_mm"], as_index=False)
             .agg(RPM=("RPM", "mean"), drag=("drag", "mean")))

    # Build one curve per (Cell_Label, RPM)
    series = []
    for (label, rpm_key), d in grp.groupby(["Cell_Label", "RPM_key"], sort=True):
        d = d.sort_values("h_mm")
        if len(d) >= 4 and np.isfinite(d["drag"]).all():
            series.append((label, float(d["RPM"].mean()), d["h_mm"].to_numpy(), d["drag"].to_numpy()))

    if not series:
        print(f"No usable drag curves for sample: {plot_name}")
        return

    # If there are many curves (e.g., silicone), keep a representative subset
    if len(series) > max_series:
        rpms_all = np.array([s[1] for s in series], dtype=float)
        order = np.argsort(rpms_all)
        pick = np.linspace(0, len(order) - 1, max_series).round().astype(int)
        series = [series[order[i]] for i in pick]

    fk = figure_key if figure_key is not None else f"drag_{str(plot_name).lower().replace(' ', '_')}"
    fig, ax, fid = new_subplots(figsize=(6, 5), figure_key=fk, figure_id=figure_id)
    rpms = np.array([s[1] for s in series], dtype=float)
    if len(np.unique(rpms)) > 1:
        norm = mpl.colors.Normalize(vmin=float(np.min(rpms)), vmax=float(np.max(rpms)))
    else:
        norm = None

    for i, (label, rpm, h, drag) in enumerate(series):
        color = GOOGLE_CMAP(norm(rpm)) if norm is not None else GOOGLE_COLORS[i % len(GOOGLE_COLORS)]
        ax.plot(h, drag, "-", color=color, lw=1.6, alpha=0.9)
        ax.scatter(h, drag, s=32, color=color, edgecolors="k", linewidths=0.3, alpha=0.9)

    ax.set_xlabel("Re-zeroed height (mm)", fontsize=FS_LABEL)
    ax.set_ylabel("Drag  Profile", fontsize=FS_LABEL)
    ax.legend([sample], fontsize=FS_LEGEND, frameon=False, loc="best")
    _gs(ax)
    plt.tight_layout()
    plt.show()
    print(f"Figure ID [{fk}] = {fid}")

In [ ]:
plot_drag_vs_height_sample("silicone", max_series=14, figure_key="drag_silicone")

In [ ]:
plot_drag_vs_height_sample("PEG", max_series=14, figure_key="drag_peg")

In [ ]:
plot_drag_vs_height_sample("sepineo", max_series=14, figure_key="drag_sepineo")

In [ ]:
plot_drag_vs_height_sample("solagum", max_series=14, figure_key="drag_solagum")

In [ ]:
plot_drag_vs_height_sample("carbopol", max_series=14, figure_key="drag_carbopol")

In [ ]:
SYY_CSV = str(DATA_DIR / "syy.csv")
plot_drag_vs_height_sample("SYY", csv_path=SYY_CSV, display_name="SYY", max_series=14, figure_key="drag_syy")

In [ ]:
SEKISUI_B41_CSV = str(DATA_DIR / "sekisui_b41.csv")
plot_drag_vs_height_sample("Sekisui-B41", csv_path=SEKISUI_B41_CSV, display_name="Sekisui-B41", max_series=14, figure_key="drag_sekisui_b41")

In [ ]:
HY500_CSV = str(DATA_DIR / "hy500.csv")
plot_drag_vs_height_sample("HY-500", csv_path=HY500_CSV, display_name="HY-500", max_series=14, figure_key="drag_hy500")